# Phase 1: Exploratory Data Analysis (EDA) on Open Food Facts

**Objective:** Query `data/raw/food.parquet` directly in place using DuckDB without loading 7.9 GB into RAM. Profile data quality, understand nested schemas, analyze country distributions, and establish the subset filtering rule.

In [ ]:
import sys
import io
from pathlib import Path
import duckdb

# Ensure UTF-8 output on Windows
if sys.platform == "win32" and isinstance(sys.stdout, io.TextIOWrapper):
    sys.stdout.reconfigure(encoding="utf-8")

# Connect to in-memory DuckDB
con = duckdb.connect()
PARQUET_PATH = '../data/raw/food.parquet'
print(f"DuckDB version: {duckdb.__version__}")

## 1. High-Level Summary & Schema Inspection

In [ ]:
# Total record count and baseline completeness
con.sql(f'''
    SELECT 
        count(*) AS total_records,
        count(code) AS has_code,
        count(product_name) AS has_name_column,
        count(ingredients_text) AS has_ingredients_column
    FROM '{PARQUET_PATH}'
''').show()

## 2. Inspecting Nested Column Types (`product_name`, `ingredients_text`, `nutriments`)

In [ ]:
# Sample nested structure for product name and ingredients
con.sql(f'''
    SELECT 
        code,
        product_name[1].lang AS first_name_lang,
        product_name[1].text AS first_name_text,
        len(ingredients_text) AS num_ingredient_translations,
        len(nutriments) AS num_nutrients_recorded
    FROM '{PARQUET_PATH}'
    WHERE len(product_name) > 0 AND len(ingredients_text) > 0
    LIMIT 5
''').show()

## 3. Geographic Distribution: Top 10 Countries

In [ ]:
# Top 10 countries represented in dataset
con.sql(f'''
    SELECT 
        list_extract(countries_tags, 1) AS country,
        count(*) AS product_count,
        round(count(*) * 100.0 / sum(count(*)) over (), 2) AS pct_share
    FROM '{PARQUET_PATH}'
    WHERE len(countries_tags) > 0
    GROUP BY country
    ORDER BY product_count DESC
    LIMIT 10
''').show()

## 4. Completeness Distribution

In [ ]:
# Completeness distribution bins (0.0 to 1.0)
con.sql(f'''
    SELECT 
        floor(completeness * 10) / 10 AS completeness_bin,
        count(*) AS count,
        round(count(*) * 100.0 / (SELECT count(*) FROM '{PARQUET_PATH}'), 2) AS pct
    FROM '{PARQUET_PATH}'
    GROUP BY completeness_bin
    ORDER BY completeness_bin DESC
''').show()

## 5. Subset Filtering Funnel Analysis

Measuring the drop at each quality gate to isolate our target ~200k high-quality products.

In [ ]:
# Filter funnel evaluation
con.sql(f'''
    SELECT 
        count(*) AS raw_total,
        count(CASE WHEN regexp_matches(code, '^[0-9]{{8}}$|^[0-9]{{12}}$|^[0-9]{{13}}$|^[0-9]{{14}}$') THEN 1 END) AS valid_barcodes,
        count(CASE WHEN regexp_matches(code, '^[0-9]{{8}}$|^[0-9]{{12}}$|^[0-9]{{13}}$|^[0-9]{{14}}$') AND len(product_name) > 0 THEN 1 END) AS has_name,
        count(CASE WHEN regexp_matches(code, '^[0-9]{{8}}$|^[0-9]{{12}}$|^[0-9]{{13}}$|^[0-9]{{14}}$') AND len(product_name) > 0 AND len(ingredients_text) > 0 THEN 1 END) AS has_ingredients,
        count(CASE WHEN regexp_matches(code, '^[0-9]{{8}}$|^[0-9]{{12}}$|^[0-9]{{13}}$|^[0-9]{{14}}$') AND len(product_name) > 0 AND len(ingredients_text) > 0 AND len(images) > 0 THEN 1 END) AS has_images,
        count(CASE WHEN regexp_matches(code, '^[0-9]{{8}}$|^[0-9]{{12}}$|^[0-9]{{13}}$|^[0-9]{{14}}$') AND len(product_name) > 0 AND len(ingredients_text) > 0 AND len(images) > 0 AND len(nutriments) > 0 AND completeness >= 0.5 THEN 1 END) AS high_quality_complete
    FROM '{PARQUET_PATH}'
''').show()